# Module 4: Machine Learning Fundamentals

**BigData Analytics Course — AKTN Magister Program**

---

## Learning Objectives
1. Understand the ML workflow: problem framing → feature engineering → modelling → evaluation.
2. Implement **regression** (Linear, Ridge, Random Forest) and evaluate with RMSE / R².
3. Implement **classification** (Logistic Regression, Decision Tree, Random Forest) and evaluate with accuracy, precision, recall, F1.
4. Implement **clustering** (K-Means, DBSCAN) for unsupervised learning.
5. Apply **cross-validation** and **hyperparameter tuning** with `GridSearchCV`.

**Estimated time:** 150 minutes  
**Prerequisites:** Modules 1–3

In [ ]:
# All libraries are pre-installed on Google Colab
# !pip install scikit-learn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (
    mean_squared_error, r2_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.decomposition import PCA
from sklearn.datasets import load_breast_cancer, fetch_california_housing
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print("✅ Ready!")

## 1. Machine Learning Taxonomy

```
Machine Learning
├── Supervised Learning
│   ├── Regression        → predict a continuous value
│   └── Classification    → predict a discrete label
├── Unsupervised Learning
│   ├── Clustering        → discover groups in data
│   ├── Dimensionality Reduction → compress features
│   └── Anomaly Detection → find outliers
└── Reinforcement Learning → learn from environment rewards
```

## 2. Regression — Predicting House Prices

We use the **California Housing** dataset (~20,000 samples).

In [ ]:
# ── 2.1 Load and explore ─────────────────────────────────────────────────────
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target   # y = median house value (100k$)

print("Feature names:", list(X.columns))
print(f"Shape: {X.shape}")
print("\nFirst 5 rows:")
display(X.head())
print("\nTarget statistics:")
print(y.describe().round(3))

In [ ]:
# ── 2.2 Train / test split and feature scaling ──────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# ── 2.3 Compare regression models ───────────────────────────────────────────
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1.0)':     Ridge(alpha=1.0),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    X_tr = X_train_sc if name != 'Random Forest' else X_train.values
    X_te = X_test_sc  if name != 'Random Forest' else X_test.values
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    results[name] = {'RMSE': round(rmse, 4), 'R²': round(r2, 4)}
    print(f"{name:<25} RMSE={rmse:.4f}  R²={r2:.4f}")

best_model = models['Random Forest']

In [ ]:
# ── 2.4 Visualise predictions ────────────────────────────────────────────────
y_pred_rf = best_model.predict(X_test.values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_rf, alpha=0.3, s=10, color='#2196F3')
lims = [min(y_test.min(), y_pred_rf.min()),
        max(y_test.max(), y_pred_rf.max())]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual Value')
axes[0].set_ylabel('Predicted Value')
axes[0].set_title('Random Forest — Actual vs Predicted')
axes[0].legend()

# Feature importances
importances = pd.Series(best_model.feature_importances_, index=X.columns)
importances.sort_values().plot.barh(ax=axes[1], color='#4CAF50', edgecolor='white')
axes[1].set_title('Feature Importances')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 3. Classification — Cancer Diagnosis

We use the **Breast Cancer Wisconsin** dataset (569 samples, 30 features, binary label).

In [ ]:
# ── 3.1 Load dataset ─────────────────────────────────────────────────────────
cancer = load_breast_cancer(as_frame=True)
X_c, y_c = cancer.data, cancer.target
print(f"Classes: {cancer.target_names}")
print(f"Class distribution:\n{y_c.value_counts().to_string()}")
print(f"Shape: {X_c.shape}")

In [ ]:
# ── 3.2 Pipeline: scale + classify ──────────────────────────────────────────
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_c, y_c, test_size=0.25, random_state=42, stratify=y_c
)

classifiers = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Decision Tree': Pipeline([
        ('clf', DecisionTreeClassifier(max_depth=5, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
}

for name, clf in classifiers.items():
    clf.fit(X_tr_c, y_tr_c)
    acc = clf.score(X_te_c, y_te_c)
    cv  = cross_val_score(clf, X_c, y_c, cv=5, scoring='accuracy')
    print(f"{name:<22} Test Acc={acc:.4f}  CV Acc={cv.mean():.4f} ± {cv.std():.4f}")

In [ ]:
# ── 3.3 Detailed evaluation — best classifier ──────────────────────────────
best_clf = classifiers['Random Forest']
y_pred_c = best_clf.predict(X_te_c)

print("Classification Report:")
print(classification_report(y_te_c, y_pred_c,
                             target_names=cancer.target_names))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_te_c, y_pred_c,
    display_labels=cancer.target_names,
    cmap='Blues', ax=ax
)
ax.set_title('Confusion Matrix — Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Hyperparameter tuning with GridSearchCV ─────────────────────────────
param_grid = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth':    [None, 5, 10],
}

rf_pipe = Pipeline([('clf', RandomForestClassifier(random_state=42))])
grid_search = GridSearchCV(rf_pipe, param_grid, cv=5,
                           scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_tr_c, y_tr_c)

print("Best params :", grid_search.best_params_)
print("Best CV F1  :", round(grid_search.best_score_, 4))
print("Test F1     :", round(
    cross_val_score(grid_search.best_estimator_, X_c, y_c,
                    cv=5, scoring='f1_weighted').mean(), 4
))

## 4. Unsupervised Learning — Clustering

In [ ]:
# ── 4.1 Generate synthetic 2-D cluster data ──────────────────────────────────
from sklearn.datasets import make_blobs

X_cl, y_cl = make_blobs(n_samples=600, centers=4,
                         cluster_std=1.2, random_state=42)

# K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init='auto')
labels_km = kmeans.fit_predict(X_cl)

# DBSCAN
dbscan = DBSCAN(eps=1.5, min_samples=10)
labels_db = dbscan.fit_predict(X_cl)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, labels, title in zip(
    axes,
    [y_cl, labels_km, labels_db],
    ['True Labels', 'K-Means (k=4)', 'DBSCAN']
):
    scatter = ax.scatter(X_cl[:, 0], X_cl[:, 1],
                         c=labels, cmap='tab10', s=20, alpha=0.7)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

# Mark K-Means centroids
axes[1].scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    c='red', marker='X', s=200, zorder=5, label='Centroids'
)
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"DBSCAN found {len(set(labels_db)) - (1 if -1 in labels_db else 0)} clusters + {(labels_db==-1).sum()} noise points")

In [ ]:
# ── 4.2 Elbow method — choosing k ───────────────────────────────────────────
inertias = []
k_range = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km.fit(X_cl)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), inertias, 'o-', color='#2196F3', linewidth=2)
ax.axvline(4, color='red', ls='--', label='Optimal k=4')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia (Within-Cluster Sum of Squares)')
ax.set_title('Elbow Method for Optimal k')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Dimensionality Reduction — PCA

In [ ]:
# ── PCA on the cancer dataset (30 → 2 dimensions for visualisation) ─────────
scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X_c)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance ratio: {pca.explained_variance_ratio_.round(3)}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

fig, ax = plt.subplots(figsize=(8, 6))
for label, name, color in zip([0, 1], cancer.target_names, ['#F44336', '#4CAF50']):
    mask = y_c == label
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               label=name, color=color, alpha=0.6, s=25)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Breast Cancer Dataset — PCA 2-D Projection')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Summary

| Task | Algorithm(s) | Key Metrics |
|------|-------------|-------------|
| Regression | Linear, Ridge, Random Forest | RMSE, R² |
| Classification | Logistic, Decision Tree, Random Forest | Accuracy, Precision, Recall, F1 |
| Clustering | K-Means, DBSCAN | Inertia, Silhouette score |
| Dim. Reduction | PCA | Explained variance ratio |

**Scikit-learn API pattern:**
```python
model = SomeModel(hyperparams)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
score  = metric(y_test, y_pred)
```

## 📝 Exercises

1. **Regression:** Train a **Gradient Boosting Regressor** on the California Housing dataset and compare it with the Random Forest results.
2. **Classification:** Use `StratifiedKFold` with 10 folds on the cancer dataset and report average AUC-ROC for each classifier.
3. **Clustering:** Apply K-Means clustering to the e-commerce dataset from Module 2, using `customer_age`, `quantity`, and `revenue` as features. Interpret the resulting clusters.

---
⬅️ [Module 3 — Data Visualization](Module_03_Data_Visualization.ipynb)  
➡️ [Module 5 — Apache Spark & PySpark](Module_05_Apache_Spark_PySpark.ipynb)